# YOLOv8 Evaluation — AI Racing Tech (single-class `car`)

Evaluates a fine-tuned YOLOv8-seg checkpoint against a held-out set of SAM2-labeled
rosbags. Supports both prime-lens and fisheye-lens camera views and expects
single-class (`car`) labels.

**Inputs**
- `DATASETS_DIR`: directory containing held-out bags in the format
  `bag/images/*.jpg` and `bag/labels/*.txt` (YOLO segmentation labels,
  class id `0`). If you only have SAM2 masks, run `sam2_finetune_med.py --format_only`
  first to generate labels.
- `DATA_YAML`: path to `testing_data.yaml` (single class, `0: car`).
- `MODEL_PATH`: path to the fine-tuned `.pt` or `.onnx` weights.

**Output**
A flat `eval_data/eval/{images,labels}/` directory built from all bags, followed
by `model.val()` metrics (box + mask mAP).


In [ ]:
import os
import random
import shutil

import torch
import tqdm
import ultralytics
from ultralytics import YOLO

## Config — edit these paths for your machine

In [ ]:
# Absolute or relative paths. Defaults assume the layout documented in the README:
#   <workspace>/YOLOv8-Fine-Tune/   <- this repo
#   <workspace>/holdout_bags/       <- held-out SAM2-labeled bags
#   <workspace>/eval_data/          <- generated by this notebook
#
# After training, Ultralytics saves the best checkpoint to:
#   src/runs/segment/<run_name>/weights/best.pt
# Update MODEL_PATH below to match the specific run directory printed by sam2_finetune_med.py.
CURR_DIR = os.getcwd()                       # .../YOLOv8-Fine-Tune/src
WORKSPACE_DIR = os.path.dirname(CURR_DIR)    # .../YOLOv8-Fine-Tune
PARENT_DIR = os.path.dirname(WORKSPACE_DIR)  # .../

DATASETS_DIR = os.path.join(PARENT_DIR, 'holdout_bags')
EVAL_OUT_DIR = os.path.join(PARENT_DIR, 'eval_data')
DATA_YAML    = os.path.join(WORKSPACE_DIR, 'testing_data.yaml')
# Replace <run_name> with the actual run directory, e.g.:
#   yolov8m-img_size_1056_layers_frozen_10_2025-04-07-01-33-24
MODEL_PATH   = os.path.join(CURR_DIR, 'runs', 'segment', '<run_name>', 'weights', 'best.pt')

IMG_SIZE = 1056  # must match training imgsz

# Keep empty (no-car) frames so precision is measured honestly.
KEEP_EMPTY_FRAMES = True
PERCENTAGE_EMPTY_FRAMES_TO_KEEP = 1.0

# Per-dataset frame weighting (name substring -> keep probability or duplication factor).
DATASET_WEIGHTS = {}

for p in [DATASETS_DIR, DATA_YAML, MODEL_PATH]:
    print(p, '->', 'OK' if os.path.exists(p) else 'MISSING')

## Build flat eval dataset from held-out bags

In [ ]:
def _generate_empty_label(path):
    with open(path, 'w') as f:
        f.write('')

def _choose_weight(img_src, weighted, removed):
    dataset_name = os.path.basename(os.path.dirname(os.path.dirname(img_src)))
    for key, w in DATASET_WEIGHTS.items():
        if key in dataset_name:
            if w > 1:
                weighted[key] = weighted.get(key, 0) + (w - 1)
                return int(w)
            if random.random() < w:
                return 1
            removed[key] = removed.get(key, 0) + 1
            return 0
    return 1

def _copy_pair(label_src, img_src, label_dst, img_dst, counters):
    weighted, removed = counters['weighted'], counters['removed']
    img_stem   = os.path.splitext(img_dst)[0]
    label_stem = os.path.splitext(label_dst)[0]
    if not os.path.exists(label_src):
        if not KEEP_EMPTY_FRAMES or random.random() >= PERCENTAGE_EMPTY_FRAMES_TO_KEEP:
            return
        counters['empty'] += 1
        for i in range(_choose_weight(img_src, weighted, removed)):
            shutil.copy(img_src, f'{img_stem}_{i}.jpg')
            _generate_empty_label(f'{label_stem}_{i}.txt')
        return
    # Normalize any legacy class ids to 0 (single-class car) without mutating the source.
    with open(label_src) as f:
        lines = f.readlines()
    fixed = []
    for line in lines:
        parts = line.strip().split()
        if not parts:
            continue
        parts[0] = '0'
        fixed.append(' '.join(parts) + '\n')
    for i in range(_choose_weight(img_src, weighted, removed)):
        shutil.copy(img_src, f'{img_stem}_{i}.jpg')
        with open(f'{label_stem}_{i}.txt', 'w') as f:
            f.writelines(fixed)

def format_eval_dataset(datasets_dir, out_dir, data_yaml):
    assert os.path.exists(datasets_dir), f'Missing {datasets_dir}'
    assert os.path.exists(data_yaml),    f'Missing {data_yaml}'

    pairs = []
    for bag in sorted(os.listdir(datasets_dir)):
        bag_imgs = os.path.join(datasets_dir, bag, 'images')
        bag_lbls = os.path.join(datasets_dir, bag, 'labels')
        if not os.path.isdir(bag_imgs):
            continue
        for img in sorted(os.listdir(bag_imgs)):
            label = os.path.join(bag_lbls, os.path.splitext(img)[0] + '.txt')
            pairs.append((os.path.join(bag_imgs, img), label, bag))

    random.seed(0)
    random.shuffle(pairs)

    if os.path.exists(out_dir):
        print(f'Deleting existing {out_dir}/')
        shutil.rmtree(out_dir)
    os.makedirs(os.path.join(out_dir, 'eval', 'images'))
    os.makedirs(os.path.join(out_dir, 'eval', 'labels'))

    counters = {'empty': 0, 'weighted': {}, 'removed': {}}
    kept = 0
    for uid, (img_src, label_src, bag) in enumerate(tqdm.tqdm(pairs, desc='Copying')):
        stem = os.path.splitext(os.path.basename(img_src))[0]
        img_dst   = os.path.join(out_dir, 'eval', 'images', f'{bag}_{stem}_{uid}.jpg')
        label_dst = os.path.join(out_dir, 'eval', 'labels', f'{bag}_{stem}_{uid}.txt')
        _copy_pair(label_src, img_src, label_dst, img_dst, counters)
        kept += 1

    shutil.copy(data_yaml, out_dir)
    print(f'Eval frames:        {kept}')
    print(f'Empty frames kept:  {counters["empty"]}')
    print(f'Weighted additions: {counters["weighted"]}')
    print(f'Removed frames:     {counters["removed"]}')

format_eval_dataset(DATASETS_DIR, EVAL_OUT_DIR, DATA_YAML)

## Run validation

In [ ]:
print('CUDA available:', torch.cuda.is_available())
print('Torch CUDA:    ', torch.version.cuda)

model = YOLO(MODEL_PATH, task='segment')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

results = model.val(data=DATA_YAML, imgsz=IMG_SIZE)
print(results.results_dict)

## Per-lens-type evaluation
Tag each bag (or bag name substring) with its lens type to get separate mAP numbers
for prime-lens and fisheye cameras. A model that achieves 0.85 aggregate mAP might be
0.95 on prime and 0.65 on fisheye — the fisheye number is what matters on track.

In [ ]:
import tempfile
from collections import defaultdict

# Map bag-name substrings to lens type labels.
# Any bag whose directory name contains a key will be tagged with the corresponding label.
# Bags that match no key are grouped under 'untagged'.
LENS_TYPE_TAGS = {
    # 'ims_run':   'prime',
    # 'fisheye':   'fisheye',
}

eval_imgs_dir = os.path.join(EVAL_OUT_DIR, 'eval', 'images')
eval_lbls_dir = os.path.join(EVAL_OUT_DIR, 'eval', 'labels')

lens_groups = defaultdict(list)
for fname in sorted(os.listdir(eval_imgs_dir)):
    lens_type = 'untagged'
    for tag, ltype in LENS_TYPE_TAGS.items():
        if tag in fname:
            lens_type = ltype
            break
    lens_groups[lens_type].append(fname)

print('Frame counts by lens type:')
for ltype, frames in sorted(lens_groups.items()):
    print(f'  {ltype}: {len(frames)} frames')

# Run model.val() on each tagged lens-type subset using a temporary directory.
# 'untagged' bags are skipped — populate LENS_TYPE_TAGS above to include them.
for lens_type, fnames in sorted(lens_groups.items()):
    if lens_type == 'untagged':
        print(f'\nSkipping {len(fnames)} untagged frames. Add entries to LENS_TYPE_TAGS to evaluate them.')
        continue
    print(f'\n--- {lens_type} lens ({len(fnames)} frames) ---')
    with tempfile.TemporaryDirectory() as tmp:
        tmp_imgs = os.path.join(tmp, 'images')
        tmp_lbls = os.path.join(tmp, 'labels')
        os.makedirs(tmp_imgs)
        os.makedirs(tmp_lbls)
        for fname in fnames:
            shutil.copy(os.path.join(eval_imgs_dir, fname), os.path.join(tmp_imgs, fname))
            lbl = os.path.splitext(fname)[0] + '.txt'
            lbl_src = os.path.join(eval_lbls_dir, lbl)
            if os.path.exists(lbl_src):
                shutil.copy(lbl_src, os.path.join(tmp_lbls, lbl))
            else:
                with open(os.path.join(tmp_lbls, lbl), 'w'):
                    pass  # empty label for no-car frames
        subset_yaml = os.path.join(tmp, 'subset.yaml')
        with open(subset_yaml, 'w') as f:
            f.write(f'val: {tmp_imgs}\n')
            f.write('nc: 1\n\nnames:\n  0: car\n')
        r = model.val(data=subset_yaml, imgsz=IMG_SIZE)
        rd = r.results_dict
        print(f'  box  mAP50:    {rd.get("metrics/mAP50(B)",   float("nan")):.4f}')
        print(f'  box  mAP50-95: {rd.get("metrics/mAP50-95(B)",float("nan")):.4f}')
        print(f'  mask mAP50:    {rd.get("metrics/mAP50(M)",   float("nan")):.4f}')
        print(f'  mask mAP50-95: {rd.get("metrics/mAP50-95(M)",float("nan")):.4f}')


## Notes
- `MODEL_PATH` can point to either a `.pt` or an `.onnx` file. For ONNX,
  `imgsz` must match the export size (`sam2_finetune_med.py` exports at
  `IMG_SIZE = 1056`).
- `testing_data.yaml` must have `nc: 1` and `names: {0: car}` — the label
  normalization step above rewrites any legacy class ids to `0`.
- Results are written to `runs/segment/val*/` under the current working dir.
